# 03. Speech embeddings for the DW-NOMINATE regression (H5)

Embeds every eligible turn with `BAAI/bge-base-en-v1.5` (truncated at 512 tokens) and averages the embeddings per member and Congress (`docs/preregistration.md` §6, H5). Eligible means a turn of 50+ words by a member in the Voteview file with a DW-NOMINATE score, not a delegate, not a party switcher, and not removed by the template filter; independents are included. Members with fewer than 10 eligible turns in a Congress are dropped. The ridge regression itself runs on a CPU (`scripts/07_nominate.py`).

Output: `embeddings/member_congress_bge.parquet`, logged to MLflow and uploaded to the private dataset.

**Before running:** L4 GPU; the same secrets. Expect 30–45 minutes, about 3–4 compute units.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L

## 2. Get the code
The repository is private until the post is published, so the clone uses the `GITHUB_TOKEN` secret if it is set.

In [ ]:
import os, subprocess, sys
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

token = secret("GITHUB_TOKEN")
url = f"https://{'kvenanzi:' + token + '@' if token else ''}github.com/kvenanzi/congress-speech.git"   # fine-grained PAT, Contents: read
if not os.path.isdir("congress-speech"):
    subprocess.run(["git", "clone", "-q", url], check=True)
else:
    subprocess.run(["git", "-C", "congress-speech", "pull", "-q"], check=True)
sys.path.insert(0, "congress-speech")
!git -C congress-speech log --oneline -1

## 3. Install what Colab lacks
MLflow is pinned to the server's version.

In [ ]:
%pip install -q "mlflow==3.14.0" optuna

## 4. Connect to MLflow and Hugging Face
Read the compute-unit rate for this runtime from Colab's resources panel and type it below; it is stored on every run as `cu_estimate`.

In [ ]:
CU_PER_HOUR = None   # e.g. 4.8 for an L4: read it from the resources panel
assert CU_PER_HOUR, "set CU_PER_HOUR from Colab's resources panel"

os.environ["MLFLOW_TRACKING_URI"] = "https://mlflow.withinnoise.dev"
os.environ["MLFLOW_TRACKING_USERNAME"] = userdata.get("MLFLOW_USER")
os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("MLFLOW_PASSWORD")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

import torch, transformers
from congress_speech import tracking, finetune, stats
mlflow = tracking.setup()
COMMIT = subprocess.run(["git", "-C", "congress-speech", "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
RUNTIME = "colab-" + torch.cuda.get_device_name(0).split()[-1]
print(RUNTIME, "torch", torch.__version__, "transformers", transformers.__version__, "commit", COMMIT)

## 5. Eligible turns

In [ ]:
import pandas as pd, numpy as np, glob
from huggingface_hub import snapshot_download
local = snapshot_download("kvenanzi/congressional-record-speeches-1995-2026", repo_type="dataset", allow_patterns=["turns/*.parquet"])
cols = ["bioguide", "congress", "party", "chamber", "text", "n_words", "long", "in_voteview", "switcher",
        "delegate", "template", "nominate_dim1"]
t = pd.concat([pd.read_parquet(f, columns=cols) for f in sorted(glob.glob(local + "/turns/*.parquet"))])
t = t[t.long & t.in_voteview & ~t.switcher & ~t.delegate & ~t.template & t.nominate_dim1.notna()]
n = t.groupby(["bioguide", "congress"]).size()
keep = n[n >= 10].index
t = t.set_index(["bioguide", "congress"]).loc[keep].reset_index()
print(len(t), "turns,", len(keep), "member-Congresses")

## 6. Embed and average

In [ ]:
from sentence_transformers import SentenceTransformer
import time
enc = SentenceTransformer("BAAI/bge-base-en-v1.5", device="cuda")
enc.max_seq_length = 512
enc.half()
t0 = time.time()
E = enc.encode(t.text.tolist(), batch_size=128, normalize_embeddings=True, show_progress_bar=True,
               convert_to_numpy=True).astype(np.float32)
hours = (time.time() - t0) / 3600
df = pd.DataFrame(E, columns=[f"e{i}" for i in range(E.shape[1])])
df[["bioguide", "congress"]] = t[["bioguide", "congress"]].to_numpy()
mean = df.groupby(["bioguide", "congress"]).mean().reset_index()
meta = t.groupby(["bioguide", "congress"]).agg(party=("party", "first"), chamber=("chamber", "first"),
                                               nominate_dim1=("nominate_dim1", "first"), n_turns=("text", "size")).reset_index()
out = meta.merge(mean, on=["bioguide", "congress"])
out.to_parquet("/content/member_congress_bge.parquet", index=False)
print(out.shape, f"{hours*60:.0f} min")

## 7. Log and upload

In [ ]:
with mlflow.start_run(run_name="embeddings-bge-base"):
    mlflow.set_tags({"stage": "embeddings", "model": "bge-base-en-v1.5", "runtime": RUNTIME, "git_commit": COMMIT,
                     "cu_estimate": round(hours * CU_PER_HOUR, 2)})
    mlflow.log_metrics({"turns": len(t), "member_congresses": len(out), "hours": hours})
    mlflow.log_artifact("/content/member_congress_bge.parquet")
from huggingface_hub import HfApi
HfApi(token=os.environ["HF_TOKEN"]).upload_file(path_or_fileobj="/content/member_congress_bge.parquet",
    path_in_repo="embeddings/member_congress_bge.parquet", repo_id="kvenanzi/congressional-record-speeches-1995-2026", repo_type="dataset")